# V3.1 — 风险特征测试：加上 high_volatility_prob 会不会让价格模型更准？

**控制变量实验（controlled experiment）**：
- 数据：同一个风险增强版数据表（`V2.5.1_15min_Risk_Enhanced_Dataset.csv`）
- 切分：同一个按时间的 80/20 切分
- 超参数：同一组（V3 里 Optuna 找到的最优参数）
- **唯一区别**：要不要 `high_volatility_prob` 这个新特征

用 XGBoost 和 LightGBM 各做一次"有 / 无风险特征"的对比，看 MAE / RMSE / R² 有没有变好。

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# 加载"风险增强版"数据表（已经含 high_volatility_prob 这一列）
df = pd.read_csv('../data/convertData/V2.5.1_15min_Risk_Enhanced_Dataset.csv')
print('原始形状:', df.shape)

# 时区处理：混合时区（冬天+02:00 / 夏天+03:00）→ 先转 UTC 再转赫尔辛基
df['datetime'] = pd.to_datetime(df['datetime'], utc=True).dt.tz_convert('Europe/Helsinki')
df = df.sort_values('datetime').reset_index(drop=True)
print('列数:', len(df.columns))
print('是否含新特征 high_volatility_prob:', 'high_volatility_prob' in df.columns)

原始形状: (105193, 54)
列数: 54
是否含新特征 high_volatility_prob: True


In [2]:
# ═══════════════════════════════════════════════════════════════════════
# 造两套特征矩阵（控制变量法的关键）：
# - baseline（基线）：49 个原始特征，不含任何风险列
# - enhanced（增强）：49 个原始特征 + high_volatility_prob（50个）
# 注意：price_roll_std_6h 和 is_high_volatility 是"从价格算出来的答案"，
#       放进特征会泄漏信息，所以两个版本都不能要它们。
# ═══════════════════════════════════════════════════════════════════════
risk_cols = ['price_roll_std_6h', 'is_high_volatility', 'high_volatility_prob']

baseline_cols = [c for c in df.columns if c not in ['price', 'datetime'] + risk_cols]
enhanced_cols = baseline_cols + ['high_volatility_prob']

X_base = df[baseline_cols]          # 49 特征
X_enh  = df[enhanced_cols]          # 50 特征（多 high_volatility_prob）
y = df['price']

print(f'baseline: {len(baseline_cols)} 个特征')
print(f'enhanced: {len(enhanced_cols)} 个特征')

# 同一个按时间的 80/20 切分（两套特征用完全相同的行）
n = len(df)
test_size = int(n * 0.20)
train_end = n - test_size

X_base_train, X_base_test = X_base.iloc[:train_end], X_base.iloc[train_end:]
X_enh_train,  X_enh_test  = X_enh.iloc[:train_end],  X_enh.iloc[train_end:]
y_train, y_test = y.iloc[:train_end], y.iloc[train_end:]
print(f'Train: {X_base_train.shape[0]}  Test: {X_base_test.shape[0]}')

baseline: 49 个特征
enhanced: 50 个特征
Train: 84155  Test: 21038


In [3]:
# ═══════════════════════════════════════════════════════════════════════
# 用 V3 里 Optuna 找到的最优参数（固定，不再搜索）
# 两个模型、两套特征 → 共训练 4 个模型
# ═══════════════════════════════════════════════════════════════════════
xgb_params = dict(
    objective='reg:absoluteerror', n_estimators=2000,
    learning_rate=0.0590, max_depth=7, min_child_weight=21,
    subsample=0.7268, colsample_bytree=0.9753,
    reg_lambda=0.3616, reg_alpha=0.6782, random_state=42,
)

lgb_params = dict(
    objective='regression_l1', n_estimators=2000,
    learning_rate=0.0168, num_leaves=313, min_child_samples=36,
    subsample=0.9915, colsample_bytree=0.9784,
    reg_lambda=0.2826, reg_alpha=0.2350, random_state=42,
)

def make_xgb():
    return XGBRegressor(**xgb_params, verbosity=0)

def make_lgb():
    return lgb.LGBMRegressor(**lgb_params, verbose=-1)

def train_eval(make_model, Xtr, ytr, Xte, yte):
    """训练一个模型并在测试集上评估，返回 (MAE, RMSE, R²)。"""
    m = make_model()
    m.fit(Xtr, ytr)
    p = m.predict(Xte)
    return (mean_absolute_error(yte, p),
            np.sqrt(mean_squared_error(yte, p)),
            r2_score(yte, p))

results = {
    'XGBoost baseline':  train_eval(make_xgb, X_base_train, y_train, X_base_test, y_test),
    'XGBoost +risk':     train_eval(make_xgb, X_enh_train,  y_train, X_enh_test,  y_test),
    'LightGBM baseline': train_eval(make_lgb, X_base_train, y_train, X_base_test, y_test),
    'LightGBM +risk':    train_eval(make_lgb, X_enh_train,  y_train, X_enh_test,  y_test),
}

print('4 个模型训练完成 ✅')

4 个模型训练完成 ✅


In [4]:
# ═══════════════════════════════════════════════════════════════════════
# 对比表 + 结论判断
# ═══════════════════════════════════════════════════════════════════════
comp = pd.DataFrame(results, index=['MAE', 'RMSE', 'R²']).T.round(4)
print(comp)

print('\n🎯 结论判断（看 MAE，越小越好）：')
for name in ['XGBoost', 'LightGBM']:
    base_mae = comp.loc[f'{name} baseline', 'MAE']
    enh_mae  = comp.loc[f'{name} +risk', 'MAE']
    delta = enh_mae - base_mae
    if delta < 0:
        verdict = '✅ 风险特征有帮助（MAE 下降了）'
    elif delta > 0:
        verdict = '❌ 风险特征反而有害（MAE 上升了）'
    else:
        verdict = '➖ 风险特征没差别'
    print(f'{name}: MAE {base_mae:.4f} → {enh_mae:.4f}  变化 {delta:+.4f}  {verdict}')

                      MAE    RMSE      R²
XGBoost baseline   2.7555  8.0842  0.9727
XGBoost +risk      2.7957  8.2468  0.9716
LightGBM baseline  2.7165  8.1066  0.9726
LightGBM +risk     2.7426  8.3181  0.9711

🎯 结论判断（看 MAE，越小越好）：
XGBoost: MAE 2.7555 → 2.7957  变化 +0.0402  ❌ 风险特征反而有害（MAE 上升了）
LightGBM: MAE 2.7165 → 2.7426  变化 +0.0261  ❌ 风险特征反而有害（MAE 上升了）
